Démarrage de Spark

In [ ]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["HADOOP_CONF_DIR"] = os.path.expanduser("~/hadoop/etc/hadoop")

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Merge Google Form Responses")
    .master("local[*]")
    .getOrCreate()
)

spark

Lecture des deux CSV

In [ ]:
base_path = "hdfs://localhost:9000/projet/gaming_mental_health/input/gaming_mental_health.csv"
form_path = "file://" + os.path.expanduser("~/projets/gaming_mental_health/gform/form_responses.csv")

base_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path)
)

form_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(form_path)
)

print("Colonnes dataset original :")
print(base_df.columns)

print("\nColonnes Google Form :")
print(form_raw_df.columns)

print("\nNombre de lignes dataset original :", base_df.count())
print("Nombre de réponses Google Form :", form_raw_df.count())

Renommage des colonnes du gform pour correspondre au csv

In [ ]:
from pyspark.sql.functions import col

column_mapping = {
    "Age": "age",
    "Gender": "gender",
    "Average daily gaming time": "daily_gaming_hours",
    "Main game genre": "game_genre",
    "Main game played": "primary_game",
    "Main gaming platform": "gaming_platform",
    "For how many years have you been playing video games?": "years_gaming",
    "How much money do you spend on video games per month on average?": "monthly_game_spending_usd",
    "How many hours do you sleep per night on average?": "sleep_hours",
    "Sleep quality ": "sleep_quality",
    "How often is your sleep disrupted?": "sleep_disruption_frequency",
    "What is your approximate average grade out of 20?": "grade_out_of_20",
    "How would you rate your productivity in studies or work?": "work_productivity_score",
    "Which mood best describes you recently?": "mood_state",
    "How often do you experience mood swings?": "mood_swing_frequency",
    "Do you feel frustration, stress, or discomfort when you cannot play video games?": "withdrawal_symptoms",
    "Have you lost interest in other activities because of gaming?": "loss_of_other_interests",
    "Do you continue playing video games despite problems caused by gaming?": "continued_despite_problems",
    "Do you often experience eye strain related to gaming?": "eye_strain",
    "Do you often experience back or neck pain related to gaming?": "back_neck_pain",
    "What is your approximate recent weight change in kilograms?": "weight_change_kg",
    "How many hours do you exercise per week on average?": "exercise_hours_weekly",
    "How socially isolated do you feel?": "social_isolation_score",
    "How many hours do you spend socializing face-to-face per week?": "face_to_face_social_hours_weekly",
}

form_df = form_raw_df

if "Horodateur" in form_df.columns:
    form_df = form_df.drop("Horodateur")

if "Timestamp" in form_df.columns:
    form_df = form_df.drop("Timestamp")

for old_name, new_name in column_mapping.items():
    if old_name in form_df.columns:
        form_df = form_df.withColumnRenamed(old_name, new_name)

form_df.printSchema()
form_df.show(5)

Convertion des types et valeurs

In [ ]:
from pyspark.sql.functions import when, round as spark_round

# Colonnes Yes / No -> True / False
boolean_columns = [
    "withdrawal_symptoms",
    "loss_of_other_interests",
    "continued_despite_problems",
    "eye_strain",
    "back_neck_pain",
]

for c in boolean_columns:
    form_df = form_df.withColumn(
        c,
        when(col(c) == "Yes", True)
        .when(col(c) == "No", False)
        .otherwise(col(c))
    )

# Colonnes numériques
numeric_columns = [
    "age",
    "daily_gaming_hours",
    "years_gaming",
    "monthly_game_spending_usd",
    "sleep_hours",
    "grade_out_of_20",
    "work_productivity_score",
    "weight_change_kg",
    "exercise_hours_weekly",
    "social_isolation_score",
    "face_to_face_social_hours_weekly",
]

for c in numeric_columns:
    form_df = form_df.withColumn(c, col(c).cast("double"))

# Conversion note /20 -> GPA /4
form_df = form_df.withColumn(
    "grades_gpa",
    spark_round(col("grade_out_of_20") / 5, 2)
)

form_df = form_df.drop("grade_out_of_20")

form_df.printSchema()
form_df.show(5)

Génération d'une colonne pas présente dans le gform mais présente dans le csv

In [ ]:
form_df = form_df.withColumn(
    "academic_work_performance",
    when((col("grades_gpa") >= 3.4) & (col("work_productivity_score") >= 8), "Excellent")
    .when((col("grades_gpa") >= 2.8) & (col("work_productivity_score") >= 6), "Good")
    .when((col("grades_gpa") >= 2.0) & (col("work_productivity_score") >= 4), "Average")
    .when((col("grades_gpa") >= 1.2) | (col("work_productivity_score") >= 3), "Below Average")
    .otherwise("Poor")
)

form_df.select(
    "grades_gpa",
    "work_productivity_score",
    "academic_work_performance"
).show()

Calcul du niveau de risque

In [ ]:
from pyspark.sql.functions import lit

risk_score = (
    when(col("daily_gaming_hours") >= 8, 3)
    .when(col("daily_gaming_hours") >= 5, 2)
    .when(col("daily_gaming_hours") >= 3, 1)
    .otherwise(0)
    +
    when(col("sleep_hours") < 5, 2)
    .when(col("sleep_hours") < 7, 1)
    .otherwise(0)
    +
    when(col("social_isolation_score") >= 8, 2)
    .when(col("social_isolation_score") >= 5, 1)
    .otherwise(0)
    +
    when(col("withdrawal_symptoms") == True, 1).otherwise(0)
    +
    when(col("loss_of_other_interests") == True, 1).otherwise(0)
    +
    when(col("continued_despite_problems") == True, 2).otherwise(0)
)

form_df = form_df.withColumn("risk_score", risk_score)

form_df = form_df.withColumn(
    "gaming_addiction_risk_level",
    when(col("risk_score") >= 8, "Severe")
    .when(col("risk_score") >= 5, "High")
    .when(col("risk_score") >= 3, "Moderate")
    .otherwise("Low")
)

form_df.select(
    "daily_gaming_hours",
    "sleep_hours",
    "social_isolation_score",
    "withdrawal_symptoms",
    "loss_of_other_interests",
    "continued_despite_problems",
    "risk_score",
    "gaming_addiction_risk_level"
).show()

form_df = form_df.drop("risk_score")

Génération des id de nos nouvelles entrées

In [ ]:
from pyspark.sql.functions import regexp_extract, max as spark_max, row_number, format_string
from pyspark.sql.window import Window

max_id = (
    base_df
    .select(regexp_extract(col("record_id"), r"GD(\d+)", 1).cast("int").alias("id_num"))
    .agg(spark_max("id_num").alias("max_id"))
    .collect()[0]["max_id"]
)

print("Dernier ID existant :", max_id)

window = Window.orderBy(lit(1))

form_df = (
    form_df
    .withColumn("row_num", row_number().over(window))
    .withColumn("record_id", format_string("GD%04d", col("row_num") + lit(max_id)))
    .drop("row_num")
)

form_df.select("record_id").show()

mise dans le bon ordre des colonnes

In [ ]:
form_df = form_df.select(base_df.columns)

print("Colonnes dataset original :", base_df.columns)
print("Colonnes Google Form transformé :", form_df.columns)

form_df.show(5)
form_df.printSchema()

Fusion et écriture dans HDFS

In [ ]:
merged_df = base_df.unionByName(form_df)

print("Lignes dataset original :", base_df.count())
print("Nouvelles réponses Google Form :", form_df.count())
print("Total après fusion :", merged_df.count())

output_path = "hdfs://localhost:9000/projet/gaming_mental_health/output/merged_dataset"

(
    merged_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(output_path)
)

print("Dataset fusionné écrit dans :", output_path)